# Шаг 7. Графики

Единый стиль, русские подписи, понятные числа. Функции из `app/charts.py`.

In [ ]:
# Подготовка: чтобы ноутбук видел папку app и подхватывал изменения в коде без перезапуска
import sys

sys.path.append("..")

%load_ext autoreload
%autoreload 2

In [ ]:
import json

from app.analytics import breakdown, by_period, item_detail
from app.charts import bar_chart, forecast_chart, line_chart
from app.config import SYNTHETIC_DIR
from app.data import prepare_events
from app.forecast import make_forecast, to_daily


def read_mapping(file_name: str) -> dict[str, str]:
    # Эталон разметки колонок из .json рядом с тестовым файлом
    return json.loads((SYNTHETIC_DIR / file_name).with_suffix(".json").read_text(encoding="utf-8"))


shop, _ = prepare_events(SYNTHETIC_DIR / "shop.xlsx", read_mapping("shop.xlsx"))
cafe, _ = prepare_events(SYNTHETIC_DIR / "cafe.csv", read_mapping("cafe.csv"))
salon, _ = prepare_events(SYNTHETIC_DIR / "salon.xlsx", read_mapping("salon.xlsx"))

## 1. line_chart: динамика

In [ ]:
# Было: быстрый график pandas - английские подписи, числа без пробелов
by_period(shop, "месяц").plot(title="Быстрый график", marker="o");

In [ ]:
# Стало: line_chart - русские подписи, "25 000", месяцы "янв 25", ось с нуля
line_chart(by_period(shop, "месяц"), "Выручка магазина по месяцам")

In [ ]:
# По дням: точек много, поэтому линия без кружков. Видны выходные и пики праздников
line_chart(by_period(shop, "день"), "Выручка магазина по дням")

In [ ]:
# Подпись оси подбирается сама: по штукам - "Количество, шт."
line_chart(by_period(cafe, "месяц", value="quantity"), "Кафе: продано штук по месяцам")

In [ ]:
# Динамика одного товара из item_detail - та же функция
beer = item_detail(shop, "лидское")
line_chart(beer["динамика"], f"{beer['товар']}: выручка по месяцам")

In [ ]:
# Сохранить график в файл - например, для курсовой
figure = line_chart(by_period(salon, "месяц"), "Выручка салона по месяцам")
figure.savefig("salon_monthly.png", dpi=150)
print("Сохранено: notebooks/salon_monthly.png")

## 2. bar_chart: топы из breakdown

In [ ]:
# Топ товаров магазина: лидер сверху, на каждом столбце - сумма и доля
bar_chart(breakdown(shop), "Топ товаров магазина по выручке")

In [ ]:
# Мастера салона - всего три строки, поэтому рисунок ниже
bar_chart(breakdown(salon, by="executor"), "Выручка салона по мастерам")

In [ ]:
# По дополнительному полю "Бренд" и по штукам - подпись оси меняется сама
bar_chart(breakdown(shop, by="Бренд", value="quantity", top=5), "Топ-5 брендов магазина по штукам")

## 3. forecast_chart: факт, прогноз и коридор

Сплошная линия - факт, пунктир - прогноз, полоса - вероятный диапазон (80%), вертикальная черта - "сегодня".

In [ ]:
# Магазин: прогноз на 90 дней. Заметь два пика - 7 ноября и 25 декабря:
# это праздники Беларуси, и Prophet сам предсказал по календарю, что в эти дни будет всплеск
shop_result = make_forecast(shop)
forecast_chart(to_daily(shop), shop_result["прогноз"], "Магазин: прогноз выручки на 90 дней")

In [ ]:
# Салон: история шумная - коридор широкий. Это честно: уверенности меньше
salon_result = make_forecast(salon)
forecast_chart(to_daily(salon), salon_result["прогноз"], "Салон: прогноз выручки на 30 дней")

In [ ]:
# Короткая история (45 дней): программа выбрала простой прогноз - коридора у него нет
shop_short = shop[shop["date"] >= "2026-08-17"]
short_result = make_forecast(shop_short)
forecast_chart(
    to_daily(shop_short), short_result["прогноз"], "Магазин, 45 дней истории: простой прогноз"
)